# Results Viz
Three analyses:
1. **Performance heatmap** — model × task × training paradigm
2. **Model size vs. performance** — encoder parameters vs. mean metric
3. **Release date vs. performance** — arXiv date vs. mean metric

In [ ]:
import sys
from pathlib import Path

# notebook_utils.py lives one level up (notebooks/), not in
# notebooks/visualizations/
sys.path.insert(0, str(Path(".").resolve().parent))

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.lines import Line2D

from notebook_utils import aggregate_test_metrics

SWEEPS_DIR = Path(
    "/raid/aradhachandran/thyroid/code/ultrabench2/notebooks/visualizations/runs"
)

# ── Categorical palette (dataviz skill slots 1-6, fixed order) ─────────────
# Assignment is alphabetical by model key so colours never repaint on filter.
CAT_COLORS: dict[str, str] = {
    "echocare": "#2a78d6",  # slot 1 – blue
    "samus": "#1baf7a",  # slot 2 – aqua
    "ultrasam": "#eda100",  # slot 3 – yellow
    "urfm": "#008300",  # slot 4 – green
    "usfm": "#4a3aa7",  # slot 5 – violet
    "usfmae": "#e34948",  # slot 6 – red
}

sns.set_theme(style="whitegrid", context="paper", font_scale=0.9)
plt.rcParams.update(
    {
        "axes.spines.top": False,
        "axes.spines.right": False,
        "figure.dpi": 150,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
    }
)

## Model Metadata

Update `params_m` and `arxiv_date` from the original papers.
Parameter counts below are *approximate encoder-only* totals inferred from
the architecture configs (all models are ViT-B / Swin-B scale, ≈ 86–88 M).
Dates are arXiv submission months — **verify before publishing**.

In [ ]:
MODEL_META: dict[str, dict] = {
    #          display      params (M)  arXiv date   architecture
    "echocare": {
        "display": "EchoCare",
        "params_m": 88,
        "arxiv_date": "2025-09-15",
        "arch": "Swin-B",
    },
    "samus": {
        "display": "SAMUS",
        "params_m": 86,
        "arxiv_date": "2023-09-13",
        "arch": "ViT-B",
    },
    "ultrasam": {
        "display": "UltraSAM",
        "params_m": 86,
        "arxiv_date": "2024-11-25",
        "arch": "ViT-B",
    },
    "urfm": {
        "display": "URFM",
        "params_m": 86,
        "arxiv_date": "2025-08-15",
        "arch": "ViT-B",
    },
    "usfm": {
        "display": "USFM",
        "params_m": 86,
        "arxiv_date": "2023-12-30",
        "arch": "ViT-B",
    },
    "usfmae": {
        "display": "USFMAE",
        "params_m": 86,
        "arxiv_date": "2025-10-27",
        "arch": "ViT-B",
    },
}

# Short display labels for axes
TASK_DISPLAY: dict[str, str] = {
    # Segmentation
    "aul_liver_segmentation": "AUL Liver",
    "aul_mass_segmentation": "AUL Mass",
    "busbra_segmentation": "BUS-BRA",
    "camus_segmentation": "CAMUS",
    "mmotu_segmentation": "MMOTU",
    "open_kidney_capsule_segmentation": "OK Capsule",
    "open_kidney_regions_segmentation": "OK Regions",
    "psfhs_segmentation": "PSFHS",
    "stmus_segmentation": "STMUS",
    "tn3k_segmentation": "TN3K",
    # Classification
    "aul_mass_classification": "AUL Mass",
    "busbra_histology_classification": "BUS-BRA Hist.",
    "busbra_pathology_classification": "BUS-BRA Path.",
    "butterfly_classification": "Butterfly",
    "camus_classification": "CAMUS",
    "covid_blues_classification": "COVID-BLUES",
    "fetal_planes_brain_plane_classification": "Fetal Brain",
    "fetal_planes_plane_classification": "Fetal Plane",
    "gbcu_classification": "GBCU",
    "masld_classification": "MASLD",
    "mmotu_classification": "MMOTU",
    "tn3k_classification": "TN3K",
}

MODEL_ORDER = [m for m in MODEL_META]

### Compute Exact Parameter Counts

Run this cell once to replace the placeholder `params_m` estimates in `MODEL_META` with values
computed directly from each architecture.  Checkpoint weights are **not** loaded — we patch
`_load_checkpoint` to a no-op so this works without the checkpoint files present.

In [ ]:
from unittest.mock import patch

import torch  # noqa: F401 — ensure torch is imported before backbone modules

from ultrabench.models.echocare.echocare_backbone import EchoCareBackbone
from ultrabench.models.samus.samus_backbone import SAMUSBackbone
from ultrabench.models.ultrasam.ultrasam_backbone import UltraSAMBackbone
from ultrabench.models.urfm.urfm_backbone import URFMBackbone
from ultrabench.models.usfm.usfm_backbone import USFMBackbone
from ultrabench.models.usfmae.usfmae_backbone import USFMAEBackbone

# (cls, kwargs) — checkpoint_path is a dummy string; _load_checkpoint is
# patched out
_BACKBONE_SPECS: list[tuple[str, type, dict]] = [
    (
        "echocare",
        EchoCareBackbone,
        {"checkpoint_path": "dummy", "output_intermediate_features": True},
    ),
    (
        "samus",
        SAMUSBackbone,
        {"checkpoint_path": "dummy", "output_intermediate_features": False},
    ),
    (
        "ultrasam",
        UltraSAMBackbone,
        {"checkpoint_path": "dummy", "output_intermediate_features": True},
    ),
    (
        "urfm",
        URFMBackbone,
        {
            "checkpoint_path": "dummy",
            "model_variant": "urfm_mae_vit_base_patch16",
            "output_intermediate_features": True,
        },
    ),
    (
        "usfm",
        USFMBackbone,
        {"checkpoint_path": "dummy", "output_intermediate_features": True},
    ),
    (
        "usfmae",
        USFMAEBackbone,
        {"checkpoint_path": "dummy", "output_intermediate_features": True},
    ),
]

print(f"{'Model':<12}  {'Parameters':>14}  {'(M)':>8}")
print("-" * 40)

for model_key, cls, kwargs in _BACKBONE_SPECS:
    with patch.object(cls, "_load_checkpoint", lambda self: None):
        backbone = cls(**kwargs)

    total = sum(p.numel() for p in backbone.parameters())
    params_m = total / 1e6
    MODEL_META[model_key]["params_m"] = params_m

    print(f"{MODEL_META[model_key]['display']:<12}  {total:>14,}  ({params_m:.2f} M)")

    del backbone  # free memory

print("\nMODEL_META['params_m'] updated in place.")

## Load and Process Data

In [ ]:
df_raw = aggregate_test_metrics(SWEEPS_DIR)

# Keep the latest run for each (model, task, shot_pct, seed, mode) to discard
# re-runs.
df = (
    df_raw.sort_values("datetime")
    .groupby(["model", "task", "shot_pct", "seed", "mode"], as_index=False)
    .last()
)

# Task type and primary metric.
#   Segmentation → Dice coefficient
#   Classification → Balanced accuracy
df["task_type"] = df["task"].apply(
    lambda t: "seg" if t.endswith("_segmentation") else "cls"
)
df["primary_metric"] = df["dice"].fillna(df["balanced_accuracy"])

# Training paradigm: strip the decoder prefix, keep head_only vs
# full_finetuning.
# This lets echocare (segformer) and others (upsampler) be compared on equal
# footing.
df["paradigm"] = df["mode"].apply(
    lambda m: "Head Only" if m.endswith("_head_only") else "Full Fine-tuning"
)

# Average the primary metric across seeds for each (model, task, paradigm).
df_agg = df.groupby(["model", "task", "task_type", "paradigm"], as_index=False)[
    "primary_metric"
].mean()
df_agg["task_disp"] = df_agg["task"].map(TASK_DISPLAY).fillna(df_agg["task"])

print("Models  :", sorted(df_agg["model"].unique()))
print("Tasks   :", sorted(df_agg["task"].unique()))
print(
    "Seeds / run:",
    df.groupby(["model", "task", "paradigm"])["seed"].count().describe().to_dict(),
)
df_agg.head()

## Visualization 1: Performance Heatmap

Four panels arranged as **2 paradigms × 2 task types**.
Colour encodes the primary metric on a sequential blue ramp (light = low, dark = high).
Both panels share the same 0 → 1 scale so cells are directly comparable.

In [ ]:
SEG_TASKS = sorted(t for t in df_agg["task"].unique() if t.endswith("_segmentation"))
CLS_TASKS = sorted(t for t in df_agg["task"].unique() if t.endswith("_classification"))

MODELS_PRESENT = [m for m in MODEL_ORDER if m in df_agg["model"].unique()]
MODEL_LABELS = [MODEL_META[m]["display"] for m in MODELS_PRESENT]

PARADIGMS = ["Head Only", "Full Fine-tuning"]
TASK_GROUPS = [
    ("seg", SEG_TASKS, "Segmentation (Dice)"),
    ("cls", CLS_TASKS, "Classification (Balanced Acc.)"),
]

# Width ratio proportional to number of tasks in each group
w_ratios = [len(SEG_TASKS), len(CLS_TASKS)]

fig, axes = plt.subplots(
    2,
    2,
    figsize=(
        max(14, 1.2 * (len(SEG_TASKS) + len(CLS_TASKS))),
        5.5 * len(MODELS_PRESENT) / 6,
    ),
    gridspec_kw={"width_ratios": w_ratios, "wspace": 0.08, "hspace": 0.45},
)

CMAP = sns.light_palette("#2a78d6", as_cmap=True)  # sequential blue

for row_i, paradigm in enumerate(PARADIGMS):
    for col_i, (ttype, task_list, col_title) in enumerate(TASK_GROUPS):
        ax = axes[row_i, col_i]

        subset = df_agg[
            (df_agg["paradigm"] == paradigm) & (df_agg["task_type"] == ttype)
        ]

        # Pivot: if a model ran multiple decoder variants, take the best.
        pivot = subset.pivot_table(
            index="model", columns="task", values="primary_metric", aggfunc="max"
        ).reindex(index=MODELS_PRESENT, columns=task_list)

        pivot.index = MODEL_LABELS
        pivot.columns = [TASK_DISPLAY.get(t, t) for t in task_list]

        show_cbar = col_i == 1  # colour-bar only on the right column
        sns.heatmap(
            pivot,
            ax=ax,
            cmap=CMAP,
            vmin=0.0,
            vmax=1.0,
            annot=True,
            fmt=".2f",
            annot_kws={"size": 6.5},
            linewidths=0.4,
            linecolor="#e1e0d9",
            cbar=show_cbar,
            cbar_kws={"shrink": 0.75, "label": "Metric value", "pad": 0.02}
            if show_cbar
            else {},
        )

        if row_i == 0:
            ax.set_title(col_title, fontsize=8.5, fontweight="bold", pad=6)

        # Paradigm label on the far-left column only
        if col_i == 0:
            ax.set_ylabel(paradigm, fontsize=8.5, fontweight="bold", labelpad=6)
        else:
            ax.set_ylabel("")
            ax.set_yticklabels([])

        ax.set_xlabel("")
        ax.tick_params(axis="x", rotation=40, labelsize=7)
        ax.tick_params(axis="y", rotation=0, labelsize=8)

fig.suptitle(
    "Model Performance Across Tasks and Training Paradigms",
    fontsize=11,
    fontweight="bold",
    y=1.02,
)
# plt.savefig("viz1_performance_heatmap.pdf")
plt.savefig("viz1_performance_heatmap.png")
plt.show()

## Visualization 2: Encoder Size vs. Mean Task Performance

Each model appears **twice** — circle for Head Only, diamond for Full Fine-tuning — connected by a vertical line (dumbbell) so you can read both the absolute level and the fine-tuning gain.

> **Note:** All six models are built on ViT-B / Swin-B backbones, so encoder parameter counts are nearly identical (≈ 86–88 M). Update `MODEL_META` with exact counts from each paper.

In [ ]:
# Mean performance per (model, paradigm) averaged over all tasks.
df_model = (
    df_agg.groupby(["model", "paradigm"], as_index=False)["primary_metric"]
    .mean()
    .rename(columns={"primary_metric": "mean_perf"})
)

# Separate per task-type too (for annotation)
df_model_seg = (
    df_agg[df_agg["task_type"] == "seg"]
    .groupby(["model", "paradigm"], as_index=False)["primary_metric"]
    .mean()
    .rename(columns={"primary_metric": "mean_dice"})
)
df_model_cls = (
    df_agg[df_agg["task_type"] == "cls"]
    .groupby(["model", "paradigm"], as_index=False)["primary_metric"]
    .mean()
    .rename(columns={"primary_metric": "mean_bal_acc"})
)

# Jitter models with identical param counts so dumbbells don't overlap.
param_vals = sorted({MODEL_META[m]["params_m"] for m in MODELS_PRESENT})
param_jitter = {}  # model → x position
for pval in param_vals:
    group = [m for m in MODELS_PRESENT if MODEL_META[m]["params_m"] == pval]
    offsets = np.linspace(-0.3 * (len(group) - 1), 0.3 * (len(group) - 1), len(group))
    for m, off in zip(group, offsets):
        param_jitter[m] = pval + off

fig, ax = plt.subplots(figsize=(8, 5))

for model in MODELS_PRESENT:
    color = CAT_COLORS[model]
    display = MODEL_META[model]["display"]
    x = param_jitter[model]

    perf = df_model[df_model["model"] == model].set_index("paradigm")["mean_perf"]
    y_ho = perf.get("Head Only", np.nan)
    y_ft = perf.get("Full Fine-tuning", np.nan)

    if not np.isnan(y_ho) and not np.isnan(y_ft):
        ax.plot([x, x], [y_ho, y_ft], color=color, alpha=0.35, linewidth=1.5, zorder=1)

    if not np.isnan(y_ho):
        ax.scatter(
            x,
            y_ho,
            color=color,
            s=55,
            marker="o",
            edgecolors="white",
            linewidths=0.8,
            zorder=3,
        )
    if not np.isnan(y_ft):
        ax.scatter(
            x,
            y_ft,
            color=color,
            s=70,
            marker="D",
            edgecolors="white",
            linewidths=0.8,
            zorder=3,
        )

    y_top = max(v for v in [y_ho, y_ft] if not np.isnan(v))
    ax.annotate(
        display,
        xy=(x, y_top),
        xytext=(0, 5),
        textcoords="offset points",
        ha="center",
        fontsize=7,
        color=color,
        fontweight="bold",
    )

legend_handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        color="#52514e",
        label="Head Only",
        markersize=6,
        markerfacecolor="#52514e",
        linewidth=0,
    ),
    Line2D(
        [0],
        [0],
        marker="D",
        color="#52514e",
        label="Full Fine-tuning",
        markersize=6,
        markerfacecolor="#52514e",
        linewidth=0,
    ),
]
ax.legend(handles=legend_handles, loc="lower right", fontsize=8, framealpha=0.9)

ax.set_xlabel("Approximate Encoder Parameters (M)", fontsize=9)
ax.set_ylabel("Mean Primary Metric\n(Dice for seg, Balanced Acc. for cls)", fontsize=9)
ax.set_title("Encoder Size vs. Task Performance", fontsize=11, fontweight="bold")
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.tick_params(labelsize=8)

# plt.savefig("viz2_size_vs_performance.pdf")
plt.savefig("viz2_size_vs_performance.png")
plt.show()

## Visualization 3: Release Date vs. Mean Task Performance

Same dumbbell layout as Viz 2, but with arXiv submission month on the x-axis.
Verify all dates in `MODEL_META` before citing this figure.

In [ ]:
def date_to_float(date_str: str) -> float:
    """Convert 'YYYY-MM' to a decimal year (e.g., '2024-06' → 2024.458)."""
    year, month = map(int, date_str.split("-"))
    return year + (month - 1) / 12


# Jitter models sharing the same release month
date_floats = {m: date_to_float(MODEL_META[m]["arxiv_date"]) for m in MODELS_PRESENT}
unique_dates = sorted(set(date_floats.values()))
date_jitter: dict[str, float] = {}
for dval in unique_dates:
    group = [m for m in MODELS_PRESENT if date_floats[m] == dval]
    offsets = np.linspace(
        -0.025 * (len(group) - 1), 0.025 * (len(group) - 1), len(group)
    )
    for m, off in zip(group, offsets):
        date_jitter[m] = dval + off

fig, ax = plt.subplots(figsize=(9, 5))

for model in MODELS_PRESENT:
    color = CAT_COLORS[model]
    display = MODEL_META[model]["display"]
    x = date_jitter[model]

    perf = df_model[df_model["model"] == model].set_index("paradigm")["mean_perf"]
    y_ho = perf.get("Head Only", np.nan)
    y_ft = perf.get("Full Fine-tuning", np.nan)

    if not np.isnan(y_ho) and not np.isnan(y_ft):
        ax.plot([x, x], [y_ho, y_ft], color=color, alpha=0.35, linewidth=1.5, zorder=1)

    if not np.isnan(y_ho):
        ax.scatter(
            x,
            y_ho,
            color=color,
            s=55,
            marker="o",
            edgecolors="white",
            linewidths=0.8,
            zorder=3,
        )
    if not np.isnan(y_ft):
        ax.scatter(
            x,
            y_ft,
            color=color,
            s=70,
            marker="D",
            edgecolors="white",
            linewidths=0.8,
            zorder=3,
        )

    y_top = max(v for v in [y_ho, y_ft] if not np.isnan(v))
    ax.annotate(
        display,
        xy=(x, y_top),
        xytext=(0, 5),
        textcoords="offset points",
        ha="center",
        fontsize=7,
        color=color,
        fontweight="bold",
    )

# x-axis: quarterly ticks from 2023-Q3 through 2025-Q1
tick_specs = [
    ("2023-07", "Jul '23"),
    ("2023-10", "Oct '23"),
    ("2024-01", "Jan '24"),
    ("2024-04", "Apr '24"),
    ("2024-07", "Jul '24"),
    ("2024-10", "Oct '24"),
    ("2025-01", "Jan '25"),
]
ax.set_xticks([date_to_float(d) for d, _ in tick_specs])
ax.set_xticklabels(
    [label for _, label in tick_specs], rotation=30, ha="right", fontsize=7
)

legend_handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        color="#52514e",
        label="Head Only",
        markersize=6,
        markerfacecolor="#52514e",
        linewidth=0,
    ),
    Line2D(
        [0],
        [0],
        marker="D",
        color="#52514e",
        label="Full Fine-tuning",
        markersize=6,
        markerfacecolor="#52514e",
        linewidth=0,
    ),
]
ax.legend(handles=legend_handles, loc="lower right", fontsize=8, framealpha=0.9)

ax.set_xlabel(
    "arXiv Submission Date (approximate — verify from papers)",
    fontsize=9,
)
ax.set_ylabel("Mean Primary Metric\n(Dice for seg, Balanced Acc. for cls)", fontsize=9)
ax.set_title("Release Date vs. Task Performance", fontsize=11, fontweight="bold")
ax.tick_params(axis="y", labelsize=8)

# plt.savefig("viz3_date_vs_performance.pdf")
plt.savefig("viz3_date_vs_performance.png")
plt.show()

## Appendix: Per-Task-Type Breakdown for Viz 2 & 3

Show segmentation (Dice) and classification (Macro F1) separately so the two metrics are never mixed.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

SPLIT_META: list[tuple[str, pd.DataFrame, str]] = [
    ("Segmentation (Dice)", df_model_seg, "mean_dice"),
    ("Classification (Balanced Acc.)", df_model_cls, "mean_bal_acc"),
]

for ax, (title, df_src, metric_col) in zip(axes, SPLIT_META):
    for model in MODELS_PRESENT:
        color = CAT_COLORS[model]
        display = MODEL_META[model]["display"]
        x = date_jitter[model]

        perf = df_src[df_src["model"] == model].set_index("paradigm")[metric_col]
        y_ho = perf.get("Head Only", np.nan)
        y_ft = perf.get("Full Fine-tuning", np.nan)

        if not np.isnan(y_ho) and not np.isnan(y_ft):
            ax.plot(
                [x, x], [y_ho, y_ft], color=color, alpha=0.35, linewidth=1.5, zorder=1
            )

        if not np.isnan(y_ho):
            ax.scatter(
                x,
                y_ho,
                color=color,
                s=55,
                marker="o",
                edgecolors="white",
                linewidths=0.8,
                zorder=3,
            )
        if not np.isnan(y_ft):
            ax.scatter(
                x,
                y_ft,
                color=color,
                s=70,
                marker="D",
                edgecolors="white",
                linewidths=0.8,
                zorder=3,
            )

        valid = [v for v in [y_ho, y_ft] if not np.isnan(v)]
        if valid:
            ax.annotate(
                display,
                xy=(x, max(valid)),
                xytext=(0, 5),
                textcoords="offset points",
                ha="center",
                fontsize=7,
                color=color,
                fontweight="bold",
            )

    ax.set_xticks([date_to_float(d) for d, _ in tick_specs])
    ax.set_xticklabels(
        [label for _, label in tick_specs], rotation=30, ha="right", fontsize=7
    )
    ax.set_xlabel("arXiv Date", fontsize=9)
    ax.set_ylabel(title, fontsize=9)
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.tick_params(axis="y", labelsize=8)

fig.suptitle(
    "Release Date vs. Performance — Segmentation and Classification Separately",
    fontsize=11,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
# plt.savefig("viz3b_date_vs_performance_split.pdf")
plt.savefig("viz3b_date_vs_performance_split.png")
plt.show()

---
## Analysis by Task Subgroup

Tasks are grouped into five clinically-motivated subgroups based on the use case described in the paper.
The same three analyses (heatmap, size & date scatters, cross-subgroup summary) are repeated for each group.

| Subgroup | Tasks |
|---|---|
| **Cancer Screening** | AUL mass, MMOTU, TN3K, GBCU, BUS-BRA |
| **Prenatal Care** | Fetal Planes, Fetal Brain Planes, PSFHS |
| **Anatomical Segmentation** | CAMUS seg, STMUS, OpenKidney, AUL liver |
| **Disease Diagnosis** | COVID-BLUES, MASLD |
| **Non-Clinical Applications** | CAMUS cls (quality), Butterfly |

In [ ]:
TASK_SUBGROUPS: dict[str, list[str]] = {
    "Cancer Screening": [
        "aul_mass_classification",
        "aul_mass_segmentation",
        "mmotu_classification",
        "mmotu_segmentation",
        "tn3k_classification",
        "tn3k_segmentation",
        "gbcu_classification",
        "busbra_histology_classification",
        "busbra_pathology_classification",
        "busbra_segmentation",
    ],
    "Prenatal Care": [
        "fetal_planes_plane_classification",
        "fetal_planes_brain_plane_classification",
        "psfhs_segmentation",
    ],
    "Anatomical Segmentation": [
        "camus_segmentation",
        "stmus_segmentation",
        "open_kidney_capsule_segmentation",
        "open_kidney_regions_segmentation",
        "aul_liver_segmentation",
    ],
    "Disease Diagnosis": [
        "covid_blues_classification",
        "masld_classification",
    ],
    "Non-Clinical Applications": [
        "camus_classification",
        "butterfly_classification",
    ],
}

# Categorical colours for subgroups (palette slots 1-5, same fixed order)
SUBGROUP_COLORS: dict[str, str] = {
    "Cancer Screening": "#2a78d6",
    "Prenatal Care": "#1baf7a",
    "Anatomical Segmentation": "#eda100",
    "Disease Diagnosis": "#008300",
    "Non-Clinical Applications": "#4a3aa7",
}

In [ ]:
def _dumbbell(
    ax: plt.Axes,
    df_perf: pd.DataFrame,
    x_map: dict[str, float],
) -> None:
    """Draw dumbbell markers (circle = Head Only, diamond = Full FT) on ax."""
    for model in MODELS_PRESENT:
        color = CAT_COLORS[model]
        display = MODEL_META[model]["display"]
        x = x_map[model]

        perf = df_perf[df_perf["model"] == model].set_index("paradigm")["mean_perf"]
        y_ho = perf.get("Head Only", np.nan)
        y_ft = perf.get("Full Fine-tuning", np.nan)

        valid = {
            k: v
            for k, v in {"Head Only": y_ho, "Full Fine-tuning": y_ft}.items()
            if not (isinstance(v, float) and np.isnan(v))
        }
        if not valid:
            continue

        if len(valid) == 2:
            ax.plot(
                [x, x],
                [valid["Head Only"], valid["Full Fine-tuning"]],
                color=color,
                alpha=0.35,
                linewidth=1.5,
                zorder=1,
            )
        if "Head Only" in valid:
            ax.scatter(
                x,
                valid["Head Only"],
                color=color,
                s=50,
                marker="o",
                edgecolors="white",
                linewidths=0.8,
                zorder=3,
            )
        if "Full Fine-tuning" in valid:
            ax.scatter(
                x,
                valid["Full Fine-tuning"],
                color=color,
                s=65,
                marker="D",
                edgecolors="white",
                linewidths=0.8,
                zorder=3,
            )

        ax.annotate(
            display,
            xy=(x, max(valid.values())),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center",
            fontsize=6,
            color=color,
            fontweight="bold",
        )


def _subgroup_mean(task_list: list[str]) -> pd.DataFrame:
    """Mean primary_metric per (model, paradigm) restricted to task_list."""
    present = [t for t in task_list if t in df_agg["task"].unique()]
    return (
        df_agg[df_agg["task"].isin(present)]
        .groupby(["model", "paradigm"], as_index=False)["primary_metric"]
        .mean()
        .rename(columns={"primary_metric": "mean_perf"})
    )


_DUMBBELL_LEGEND = [
    Line2D(
        [0],
        [0],
        marker="o",
        color="#52514e",
        label="Head Only",
        markersize=6,
        markerfacecolor="#52514e",
        linewidth=0,
    ),
    Line2D(
        [0],
        [0],
        marker="D",
        color="#52514e",
        label="Full Fine-tuning",
        markersize=6,
        markerfacecolor="#52514e",
        linewidth=0,
    ),
]

print("Helpers defined.")

### Subgroup Heatmaps

One figure per subgroup — same 2-paradigm × task-type layout as Viz 1, restricted to the subgroup's tasks.
Subgroups with only segmentation tasks show one column; mixed subgroups show two.

In [ ]:
for group_name, task_list in TASK_SUBGROUPS.items():
    present = [t for t in task_list if t in df_agg["task"].unique()]
    seg_tasks = [t for t in present if t.endswith("_segmentation")]
    cls_tasks = [t for t in present if t.endswith("_classification")]

    task_cols = []
    if seg_tasks:
        task_cols.append(
            (
                "seg",
                seg_tasks,
                f"Segmentation (Dice)  [{len(seg_tasks)} tasks]",
            )
        )
    if cls_tasks:
        task_cols.append(
            (
                "cls",
                cls_tasks,
                f"Classification (Bal. Acc.)  [{len(cls_tasks)} tasks]",
            )
        )

    n_cols = len(task_cols)
    widths = [len(tc[1]) for tc in task_cols]
    fig_w = max(6, 1.35 * sum(widths))
    fig_h = 2.2 + 0.45 * len(MODELS_PRESENT)

    fig, axes = plt.subplots(
        2,
        n_cols,
        figsize=(fig_w, fig_h),
        gridspec_kw={
            "width_ratios": widths if n_cols > 1 else [1],
            "wspace": 0.08,
            "hspace": 0.55,
        },
        squeeze=False,
    )

    for row_i, paradigm in enumerate(PARADIGMS):
        for col_i, (ttype, tasks, col_title) in enumerate(task_cols):
            ax = axes[row_i, col_i]

            subset = df_agg[
                (df_agg["paradigm"] == paradigm) & (df_agg["task"].isin(tasks))
            ]
            pivot = subset.pivot_table(
                index="model", columns="task", values="primary_metric", aggfunc="max"
            ).reindex(index=MODELS_PRESENT, columns=tasks)
            pivot.index = MODEL_LABELS
            pivot.columns = [TASK_DISPLAY.get(t, t) for t in tasks]

            show_cbar = col_i == n_cols - 1
            sns.heatmap(
                pivot,
                ax=ax,
                cmap=CMAP,
                vmin=0.0,
                vmax=1.0,
                annot=True,
                fmt=".2f",
                annot_kws={"size": 6.5},
                linewidths=0.4,
                linecolor="#e1e0d9",
                cbar=show_cbar,
                cbar_kws={"shrink": 0.75, "label": "Metric", "pad": 0.02}
                if show_cbar
                else {},
            )

            if row_i == 0:
                ax.set_title(col_title, fontsize=8, fontweight="bold", pad=6)
            if col_i == 0:
                ax.set_ylabel(paradigm, fontsize=8, fontweight="bold")
            else:
                ax.set_ylabel("")
                ax.set_yticklabels([])

            ax.set_xlabel("")
            ax.tick_params(axis="x", rotation=40, labelsize=7)
            ax.tick_params(axis="y", rotation=0, labelsize=8)

    fig.suptitle(
        f"{group_name} — Performance Heatmap",
        fontsize=11,
        fontweight="bold",
        y=1.02,
    )
    slug = group_name.lower().replace(" ", "_")
    plt.savefig(f"subgroup_{slug}_heatmap.png")
    plt.show()

### Subgroup Size & Date Scatters

Five rows (one per subgroup) × two columns (encoder size | release date).
Performance on the y-axis is the mean primary metric averaged over the subgroup's tasks only.
Y-axis label colour matches the subgroup colour used in the summary chart.

In [ ]:
n_groups = len(TASK_SUBGROUPS)
fig, axes = plt.subplots(
    n_groups,
    2,
    figsize=(13, 4.0 * n_groups),
    gridspec_kw={"hspace": 0.65, "wspace": 0.38},
)

for row_i, (group_name, task_list) in enumerate(TASK_SUBGROUPS.items()):
    df_perf = _subgroup_mean(task_list)

    for col_i, (xlabel, x_map) in enumerate(
        [
            ("Encoder Parameters (M)", param_jitter),
            ("arXiv Date", date_jitter),
        ]
    ):
        ax = axes[row_i, col_i]
        _dumbbell(ax, df_perf, x_map)

        if col_i == 0:
            ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
            ax.set_ylabel(
                f"{group_name}\nmean metric",
                fontsize=7.5,
                fontweight="bold",
                color=SUBGROUP_COLORS[group_name],
            )
        else:
            ax.set_xticks([date_to_float(d) for d, _ in tick_specs])
            ax.set_xticklabels(
                [label for _, label in tick_specs],
                rotation=30,
                ha="right",
                fontsize=6,
            )
            ax.set_ylabel("")

        ax.set_xlabel(xlabel, fontsize=8)
        ax.tick_params(axis="y", labelsize=7)

        if row_i == 0:
            ax.set_title(
                "Size vs. Performance"
                if col_i == 0
                else "Release Date vs. Performance",
                fontsize=9,
                fontweight="bold",
            )

fig.legend(
    handles=_DUMBBELL_LEGEND,
    loc="upper right",
    fontsize=8,
    bbox_to_anchor=(1.0, 1.0),
    framealpha=0.9,
)
fig.suptitle(
    "Model Size and Release Date vs. Performance — Stratified by Subgroup",
    fontsize=12,
    fontweight="bold",
    y=1.005,
)
plt.savefig("subgroup_size_date_scatters.png")
plt.show()

### Summary: Model Rankings Across Subgroups

One line per model tracing its mean performance across all five subgroups.
Left panel: Head Only (frozen backbone). Right panel: Full Fine-tuning.
Coloured background strips highlight each subgroup's column.

In [ ]:
# Build summary DataFrame: mean_perf per (model, paradigm, subgroup)
summary_rows = []
for group_name, task_list in TASK_SUBGROUPS.items():
    for _, row in _subgroup_mean(task_list).iterrows():
        summary_rows.append(
            {
                "group": group_name,
                "model": row["model"],
                "paradigm": row["paradigm"],
                "mean_perf": row["mean_perf"],
            }
        )
df_summary = pd.DataFrame(summary_rows)

group_names = list(TASK_SUBGROUPS)
x_pos = {g: i for i, g in enumerate(group_names)}

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=False)

for ax, paradigm in zip(axes, PARADIGMS):
    subset = df_summary[df_summary["paradigm"] == paradigm]

    for model in MODELS_PRESENT:
        color = CAT_COLORS[model]
        display = MODEL_META[model]["display"]

        model_sub = (
            subset[subset["model"] == model]
            .assign(x=lambda d: d["group"].map(x_pos))
            .sort_values("x")
        )
        ax.plot(
            model_sub["x"],
            model_sub["mean_perf"],
            color=color,
            linewidth=1.4,
            marker="o",
            markersize=5,
            markeredgecolor="white",
            markeredgewidth=0.6,
            label=display,
            zorder=3,
        )

    # Colour-coded subgroup background strips
    for g, x_val in x_pos.items():
        ax.axvspan(
            x_val - 0.45,
            x_val + 0.45,
            alpha=0.04,
            color=SUBGROUP_COLORS[g],
            zorder=0,
        )

    ax.set_xticks(list(x_pos.values()))
    ax.set_xticklabels([g.replace(" ", "\n") for g in group_names], fontsize=7.5)
    ax.set_ylabel("Mean Primary Metric\n(Dice / Balanced Acc.)", fontsize=9)
    ax.set_title(paradigm, fontsize=10, fontweight="bold")
    ax.set_xlim(-0.4, len(group_names) - 0.6)
    ax.tick_params(axis="y", labelsize=8)

    if paradigm == "Head Only":
        ax.legend(loc="upper right", fontsize=7.5, framealpha=0.9, ncol=2)

fig.suptitle(
    "Mean Performance per Subgroup — All Models",
    fontsize=12,
    fontweight="bold",
)
plt.tight_layout()
plt.savefig("subgroup_summary_lines.png")
plt.show()